# Verification of the reported numbers

Recomputes, **from the raw result files** in `CoAD_project/results/`, every number quoted in
`huong-dan-reproduce-coad.md`, `ke-hoach-reimplement-coad.md`, `huong-dan-reimplement-coad.md` and `04.md`,
with code written separately from the analysis cells. Each row: the claim, where it is quoted, the quoted value,
the recomputed value and `OK` / `MISMATCH`. Integrity checks come first (complete runs, no duplicates, files not
corrupted). CPU only, a few minutes. Result table saved to `reports/verification.csv`.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 90); pd.set_option('display.max_rows', 300)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    STORE = Path('/content/drive/MyDrive/CoAD_project')
except ImportError:
    STORE = Path('CoAD_project')          # local copy with results/ and eda/
R = STORE / 'results'
print('Result files:', sorted(p.name for p in R.glob('*.csv')))

Mounted at /content/drive
Result files: ['ablation_results_gpu-a.csv', 'm10_events_gpu-a.csv', 'm10_gradient_gpu-a.csv', 'm10_models_gpu-a.csv', 'm10_probes_gpu-a.csv', 'm10_stress_gpu-a.csv', 'm6a_smoothing_gpu-a.csv', 'm6c_results_gpu-a.csv', 'm6c_results_gpu-b.csv', 'm7_branches_gpu-a.csv', 'm8_recon_gpu-a.csv', 'm8_recon_gpu-b.csv', 'm9_events_gpu-a.csv', 'm9_models_gpu-a.csv', 'm9_stress_gpu-a.csv', 'reimpl_results_gpu-a.csv', 'reimpl_results_gpu-b.csv', 'tsbad64_results.csv', 'tsbad64_results_cpu-a.csv', 'tsbad64_results_cpu-b.csv']


**Step 0, repair (runs once, safe to re-run).** Files `m9_events*` / `m9_stress*` written by part 14 before the fix of 07/11/2026 have shifted columns for part of the models. This cell detects them and repairs them **in place on Drive**; the original is first copied to `results_backup/` (outside `results/`, so it is never read again). For every model exactly one column layout must be free of impossible values (missing peak/mean, peak < mean, values outside [0, 1], middle/edges above the peak, mean outside [middle, edges]); otherwise the file is left untouched and the cell stops. Files already correct are reported as `OK`.

In [2]:
# ---- Repair the M9 files written with shifted columns (code of part 14 before 07/11/2026), in place on Drive.
# Each model was appended with its own column order. For every model, each possible layout is tried and the one
# with no impossible value (peak percentile < mean percentile, value outside [0, 1], middle/edges present for a
# short event) is kept. The original file is first copied to results_backup/ (outside results/, so it is never read).
import csv, shutil
SPLIT = ['full', 'classify', 'reconstruct', 'backup']
EVENT_SIGNALS = SPLIT + ['full_unsmoothed', 'oe', 'residual', 'error']
E_BASE = ['run_id', 'group', 'seed', 'family', 'event', 'start', 'length', 'length_over_period', 'window_share', 'amplitude_z']
E_FULL = E_BASE + [f'{s}_{k}' for s in EVENT_SIGNALS for k in (['peak', 'mean', 'middle', 'edges'] if s in SPLIT else ['peak', 'mean'])]
E_SHORT = E_BASE + [f'{s}_{k}' for s in EVENT_SIGNALS for k in ['peak', 'mean']]
E_LATE = E_SHORT + [f'{s}_{k}' for s in SPLIT for k in ['middle', 'edges']]
S_SIG = SPLIT + ['oe']
S_BASE = ['group', 'seed', 'family', 'type', 'length_periods', 'offset', 'position']
S_FULL = S_BASE + [f'{s}_{k}' for s in S_SIG for k in ['peak', 'mean', 'middle', 'edges']] + ['spill']
S_LATE = S_BASE + [f'{s}_{k}' for s in S_SIG for k in ['peak', 'mean']] + ['spill'] + [f'{s}_{k}' for s in S_SIG for k in ['middle', 'edges']]

def to_frame(rows, cols, target):
    df = pd.DataFrame([dict(zip(c, r)) for r, c in zip(rows, cols)]).reindex(columns=target)
    for c in target:
        if c not in ('run_id', 'family', 'type'):
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

def violations(df, signals, short_mask=None):
    """Impossible values under a candidate layout: peak/mean never missing, peak >= mean, values in [0, 1],
    middle/edges missing together for every signal of a row (and missing exactly for short events, if known)."""
    pm = [c for c in df.columns if c.endswith(('_peak', '_mean'))]
    bad = int(df[pm].isna().sum().sum())
    eps = 1e-9
    for s in signals:
        peak, mean = df[f'{s}_peak'], df[f'{s}_mean']
        bad += int((peak + eps < mean).sum())
        if f'{s}_middle' in df:
            mid, edg = df[f'{s}_middle'], df[f'{s}_edges']
            has = mid.notna() & edg.notna()
            # middle and edges thirds cover the whole event: both <= peak, mean is between them
            bad += int((has & ((mid > peak + eps) | (edg > peak + eps))).sum())
            bad += int((has & ((mean < np.minimum(mid, edg) - eps) | (mean > np.maximum(mid, edg) + eps))).sum())
    values = df[[c for c in df.columns if c.endswith(('_peak', '_mean', '_middle', '_edges'))]]
    bad += int(((values < -1e-9) | (values > 1 + 1e-9)).sum().sum())
    missing = df[[c for c in df.columns if c.endswith(('_middle', '_edges'))]].isna()
    bad += int((missing.nunique(axis=1) > 1).sum())
    if short_mask is not None:
        bad += int((missing.iloc[:, 0] != short_mask).sum())
    return bad

def repair(path, layouts, target, key, signals, short_of):
    lines = list(csv.reader(open(path, newline='', encoding='utf-8')))
    header, rows = lines[0], lines[1:]
    if header != target:
        print(f'{path.name}: unexpected header, not touched'); return
    models = {}
    for r in rows:
        models.setdefault(tuple(r[i] for i in key), []).append(r)
    fixed, changed = [], 0
    for m, rs in models.items():
        valid = []
        for layout in layouts:
            if any(len(r) != len(layout) for r in rs):
                continue
            df = to_frame(rs, [layout] * len(rs), target)
            if violations(df, signals, short_of(df)) == 0:
                valid.append((layout, df))
        # exactly one layout must be free of impossible values, otherwise stop instead of guessing
        assert len(valid) == 1, f'{path.name}: model {m}: {len(valid)} possible layouts, file left untouched'
        changed += valid[0][0] != target
        fixed.append(valid[0][1])
    if changed == 0:
        print(f'{path.name}: OK, no shifted columns'); return
    backup = STORE / 'results_backup'
    backup.mkdir(exist_ok=True)
    shutil.copy2(path, backup / (path.stem + '.shifted.csv'))
    pd.concat(fixed, ignore_index=True).to_csv(path, index=False)
    print(f'{path.name}: {changed} of {len(models)} models had shifted columns -> repaired in place '
          f'(original kept in results_backup/{path.stem}.shifted.csv)')

for path in sorted(R.glob('m9_events*.csv')):
    repair(path, [E_FULL, E_LATE, E_SHORT], E_FULL, (0,), EVENT_SIGNALS, lambda df: df['length'] < 24)
for path in sorted(R.glob('m9_stress*.csv')):
    with open(path, encoding='utf-8') as fh:
        stress_header = fh.readline().strip().split(',')
    repair(path, [S_FULL, S_LATE], stress_header, (0, 1), S_SIG, lambda df: None)

m9_events_gpu-a.csv: 99 of 192 models had shifted columns -> repaired in place (original kept in results_backup/m9_events_gpu-a.shifted.csv)
m9_stress_gpu-a.csv: 21 of 30 models had shifted columns -> repaired in place (original kept in results_backup/m9_stress_gpu-a.shifted.csv)


In [3]:
def load(pattern):
    files = sorted(R.glob(pattern))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

man = pd.read_csv(STORE / 'eda' / 'manifest_tsbad64.csv')
G, fam = man['group'].tolist(), man.set_index('group')['family']
tsb = load('tsbad64_results*.csv')
tsb = tsb[tsb['status'] == 'success'].sort_values('run_id').drop_duplicates(['method', 'group'], keep='last')
def method(m, key='VUS_PR'):
    return 100 * tsb[tsb['method'] == m].set_index('group')[key].reindex(G)
ri = load('reimpl_results_*.csv')
ri = ri[ri['status'] == 'success'].sort_values('run_id').drop_duplicates(['mode', 'group', 'seed'], keep='last')
code, paper = ri[ri['mode'] == 'code'], ri[ri['mode'] == 'paper']
def seed_mean(df, key='VUS_PR'):
    per_seed = 100 * df.groupby('seed')[key].mean()
    return per_seed.mean(), per_seed.std()
def series_mean(df, key='VUS_PR', seeds=None):
    d = df if seeds is None else df[df['seed'].isin(seeds)]
    return 100 * d.groupby('group')[key].mean().reindex(G)
def holm(p):
    p = np.asarray(p); out = np.empty(len(p)); running = 0
    for i, j in enumerate(np.argsort(p)):
        running = max(running, (len(p) - i) * p[j]); out[j] = min(1, running)
    return out

checks = []   # (id, where quoted, claim, quoted value, recomputed value, tolerance)
def check(cid, where, claim, quoted, value, tol):
    checks.append((cid, where, claim, quoted, value, tol))

In [4]:
# ---- Integrity: complete runs, no duplicates, files not corrupted
raw_ri = load('reimpl_results_*.csv')
check('I01', 'plan 14', 'reimplementation: successful runs (64 x 2 modes x 3 seeds)', 384, len(ri), 0)
check('I02', 'plan 14', 'reimplementation: failed runs', 0, int((raw_ri['status'] != 'success').sum()), 0)
check('I03', 'reproduce M3', 'reproduction: methods with 64 successful series', 8, int((tsb.groupby('method')['group'].nunique() == 64).sum()), 0)
m6c = load('m6c_results*.csv'); m6c = m6c[m6c['status'] == 'success'].drop_duplicates(['config', 'group', 'seed'], keep='last')
check('I04', 'reimplement 12', 'M6-C: successful runs (8 configs x 9 MGAB x 3 seeds)', 216, len(m6c), 0)
m8 = load('m8_recon*.csv').drop_duplicates(['mode', 'group', 'seed'], keep='last')
check('I05', 'reimplement 13', 'M8: rescored models (2 modes x 64 x seeds 47-48)', 256, len(m8), 0)
ev9, mo9, st9 = load('m9_events*.csv'), load('m9_models*.csv'), load('m9_stress*.csv')
check('I06', 'reimplement 14', 'M9: analysed models', 192, mo9.drop_duplicates(['group', 'seed']).shape[0], 0)
bad = sum(int((ev9[f'{s}_peak'] + 1e-9 < ev9[f'{s}_mean']).sum()) for s in ['full', 'classify', 'reconstruct', 'backup', 'oe', 'residual', 'error'])
check('I07', 'reimplement 14', 'M9 events: rows with peak < mean percentile (0 = file not corrupted; use results_fix/ otherwise)', 0, bad, 0)
check('I08', 'reimplement 14', 'M9 events: missing OE values (0 = file not corrupted)', 0, int(ev9['oe_mean'].isna().sum()), 0)
ev10, mo10, st10, gr10 = load('m10_events*.csv'), load('m10_models*.csv'), load('m10_stress*.csv'), load('m10_gradient*.csv')
check('I09', 'reimplement 15', 'M10: analysed models', 192, mo10.drop_duplicates(['group', 'seed']).shape[0], 0)
check('I10', 'reimplement 15', 'M10: gradient runs (9 MGAB x 2 variants)', 18, gr10.drop_duplicates(['group', 'seed', 'variant']).shape[0], 0)
ab = load('ablation_results*.csv'); ab = ab[ab['status'] == 'success'].drop_duplicates(['variant', 'group'], keep='last')
check('I11', 'reproduce 10.1', 'M6-B: variants with 64 series', 3, int((ab.groupby('variant')['group'].nunique() == 64).sum()), 0)

In [5]:
# ---- Reproduction and reimplementation (section 14 of the plan, M4)
check('R01', 'reproduce M4', 'CoAD (code run) VUS-PR, 64 series', 41.70, method('CoAD').mean(), 0.006)
check('R02', 'reproduce M4', 'CoAD (code run) F1', 45.99, method('CoAD', 'f1_score').mean(), 0.006)
check('R03', 'reproduce M4', 'CoAD (code run) AUC-PR', 40.43, method('CoAD', 'auc_prc').mean(), 0.006)
m, s = seed_mean(code)
check('R04', 'plan 14.3', 'reimplementation code mode VUS-PR, mean over 3 seeds', 41.69, m, 0.006)
check('R05', 'plan 14.3', 'reimplementation code mode VUS-PR, SD over seeds', 0.96, s, 0.006)
check('R06', 'plan 14.3', 'reimplementation code mode F1', 45.62, seed_mean(code, 'f1_score')[0], 0.006)
check('R07', 'plan 14.4', 'reimplementation paper mode VUS-PR', 38.72, seed_mean(paper)[0], 0.006)
mg = man.loc[man['family'] == 'MGAB', 'group']
check('R08', 'plan 14.4', 'MGAB, code mode VUS-PR', 44.73, 100 * code[code['group'].isin(mg)]['VUS_PR'].mean(), 0.006)
check('R09', 'plan 14.4', 'MGAB, paper mode VUS-PR', 18.44, 100 * paper[paper['group'].isin(mg)]['VUS_PR'].mean(), 0.006)
check('R10', 'reproduce M3', 'MP VUS-PR, 64 series', 29.48, method('MP').mean(), 0.006)

In [6]:
# ---- M5 (part 9): ranks and pairwise tests, VUS-PR
B = ['Kmeans', 'KshapeAD', 'SAND', 'Series2Graph', 'MP', 'PCA', 'damp']
vus = pd.DataFrame({m: method(m) for m in ['CoAD'] + B})
ranks = vus.rank(axis=1, ascending=False).mean()
check('M5a', 'reimplement 9', 'average rank Kmeans', 2.61, ranks['Kmeans'], 0.006)
check('M5b', 'reimplement 9', 'average rank KshapeAD', 3.70, ranks['KshapeAD'], 0.006)
check('M5c', 'reimplement 9', 'average rank CoAD', 3.81, ranks['CoAD'], 0.006)
pv = [wilcoxon(vus['CoAD'] - vus[b]).pvalue for b in B]
check('M5d', 'reimplement 9', 'CoAD vs Kmeans: series where CoAD is better', 25, int((vus['CoAD'] > vus['Kmeans']).sum()), 0)
check('M5e', 'reimplement 9', 'CoAD vs Kmeans: Holm p (7 baselines)', 0.46, holm(pv)[0], 0.006)

In [7]:
# ---- M6-A (part 10), M6-C (part 12), M6-B (part 24 of the experiments notebook)
m6a = load('m6a_smoothing*.csv').drop_duplicates(['group', 'seed'], keep='last')
check('A01', 'plan 14.6', 'M6-A: VUS-PR without smoothing (code mode, 3 seeds)', 36.85, seed_mean(m6a)[0], 0.006)
check('A02', 'plan 14.6', 'M6-A: F1 without smoothing', 43.80, seed_mean(m6a, 'f1_score')[0], 0.006)
cfg = 100 * m6c.groupby('config')['VUS_PR'].mean()
check('C01', 'plan 14.8', 'M6-C: code - training, MGAB VUS-PR', 19.00, cfg['code - training'], 0.006)
check('C02', 'plan 14.8', 'M6-C: paper + training, MGAB VUS-PR', 29.64, cfg['paper + training'], 0.006)
check('C03', 'plan 14.8', 'M6-C: paper + architecture, MGAB VUS-PR', 10.60, cfg['paper + architecture'], 0.006)
co = tsb[tsb['method'] == 'CoAD'].set_index('group')['VUS_PR']
for cid, v, q in [('B01', 'time_soft', 1.91), ('B02', 'guide_hard', -2.40), ('B03', 'freq_time_mean', -4.86)]:
    d = ab[ab['variant'] == v].set_index('group')['VUS_PR']
    check(cid, 'reproduce 10.1 / M6-B', f'M6-B {v}: VUS-PR minus CoAD (64 series)', q, 100 * (d - co.reindex(d.index)).mean(), 0.006)

In [8]:
# ---- M7 (part 11) and M8 (part 13)
m7 = load('m7_branches*.csv')
r7 = m7[m7['branch'] == 'reconstruction only'].set_index(['mode', 'group'])['VUS_PR']
check('E01', 'reimplement 11', 'M7: paper mode, reconstruction only, VUS-PR (seed 46)', 47.61, 100 * r7.xs('paper').mean(), 0.006)
check('E02', 'reimplement 11', 'M7: code mode, full score, VUS-PR (seed 46)', 40.61, 100 * code[code['seed'] == 46]['VUS_PR'].mean(), 0.006)
rec = pd.concat([m7[m7['branch'] == 'reconstruction only'].drop(columns='branch'), m8], ignore_index=True).drop_duplicates(['mode', 'group', 'seed'], keep='last')
check('E03', 'reimplement 13', 'M8: code mode, reconstruction only, VUS-PR (3 seeds)', 42.04, seed_mean(rec[rec['mode'] == 'code'])[0], 0.006)
pr, cf = series_mean(rec[rec['mode'] == 'paper'], seeds=[47, 48]), series_mean(code, seeds=[47, 48])
check('E04', 'reimplement 13', 'M8: paper mode reconstruction only, seeds 47-48', 44.76, pr.mean(), 0.006)
check('E05', 'reimplement 13', 'M8: code mode full score, seeds 47-48', 42.23, cf.mean(), 0.006)
check('E06', 'reimplement 13', 'M8: series where the prediction holds', 41, int((pr > cf).sum()), 0)
check('E07', 'reimplement 13', 'M8: one-sided Wilcoxon p', 0.019, wilcoxon(pr - cf, alternative='greater').pvalue, 0.0006)

In [9]:
# ---- M9 (part 14)
ev = ev9.drop_duplicates(['group', 'seed', 'event'], keep='last')
found = (ev['full_peak'] >= 0.99).groupby(ev['family']).mean()
check('N01', 'reimplement 14', 'M9: share of events found, TODS', 0.021, found['TODS'], 0.0006)
check('N02', 'reimplement 14', 'M9: share of events found, IOPS', 0.421, found['IOPS'], 0.0006)
check('N03', 'reimplement 14', 'M9: share of events found, SVDB', 0.781, found['SVDB'], 0.0006)
miss = ev[ev['oe_peak'] < 0.99]
check('N04', 'reimplement 14', 'M9: events missed by OE', 5485, len(miss), 0)
check('N05', 'reimplement 14', 'M9: OE-missed events found by the raw reconstruction error', 0.531, (miss['error_peak'] >= 0.99).mean(), 0.0006)
check('N06', 'reimplement 14', 'M9: OE-missed events found by the final score', 0.071, (miss['full_peak'] >= 0.99).mean(), 0.0006)
mo = mo9.drop_duplicates(['group', 'seed'], keep='last').set_index(['group', 'seed'])
cr = 100 * code.set_index(['group', 'seed'])['VUS_PR']
for cid, v, q in [('N07', 'calibrated', 44.10), ('N08', 'calibrated + backup', 44.43)]:
    x = 100 * mo[f'{v}: VUS_PR']
    check(cid, 'reimplement 14', f'M9: {v} VUS-PR (3 seeds)', q, x.groupby(level='seed').mean().mean(), 0.006)
d = (100 * mo['calibrated: VUS_PR'] - cr.reindex(mo.index)).groupby(level='group').mean()
check('N09', 'reimplement 14', 'M9: calibrated - CoAD, series higher', 40, int((d > 0).sum()), 0)
check('N10', 'reimplement 14', 'M9: calibrated - CoAD, one-sided p (< 0.001 quoted)', 0.001, wilcoxon(d, alternative='greater').pvalue, None)
u = st9[(st9['offset'] == 0) & (st9['type'] == 'uniform')].groupby('length_periods')['full_mean'].mean()
check('N11', 'reimplement 14', 'M9 stress: uniform, 1 period minus 4 periods', 0.022, u[1.0] - u[4.0], 0.0006)

In [10]:
# ---- M10 (part 15)
e = ev10.drop_duplicates(['group', 'seed', 'event'], keep='last')
check('T01', 'reimplement 15', 'H1-A: OE middle - edges (real long events)', 0.065, (e['oe_middle'] - e['oe_edges']).mean(), 0.0006)
check('T02', 'reimplement 15', 'H1-A: number of long events', 2652, len(e), 0)
w = st10.pivot_table(index=['group', 'seed', 'position', 'type', 'amplitude'], columns='boundary', values='oe_plateau')
check('T03', 'reimplement 15', 'H1-B: OE plateau abrupt - smooth', 0.002, (w['abrupt'] - w['smooth']).mean(), 0.0006)
check('T04', 'reimplement 15', 'H1-B: pairs', 480, len(w.dropna()), 0)
m = mo10.drop_duplicates(['group', 'seed'], keep='last')
check('T05', 'reimplement 15', 'H2-A: AUC hard - soft', -0.097, (m['auc_hard'] - m['auc_soft']).mean(), 0.0006)
hb = (m['gain_anomalous'] - m['gain_normal']).dropna()
check('T06', 'reimplement 15', 'H2-B: gain anomalous - normal', -0.015, hb.mean(), 0.0006)
check('T07', 'reimplement 15', 'H2-B: models', 177, len(hb), 0)
g = gr10.drop_duplicates(['group', 'seed', 'variant'], keep='last')
check('T08', 'reimplement 15', 'Gradient: mean cosine, joint', 0.19, g.loc[g['variant'] == 'joint', 'cosine_mean'].mean(), 0.006)
wv = 100 * g.pivot_table(index='group', columns='variant', values='VUS_PR')
check('T09', 'reimplement 15', 'Gradient: VUS-PR detached - joint', -11.50, (wv['detached'] - wv['joint']).mean(), 0.006)
check('T10', 'reimplement 15', 'Gradient: series where detached is lower', 7, int((wv['detached'] < wv['joint']).sum()), 0)

In [11]:
table = pd.DataFrame(checks, columns=['id', 'quoted in', 'claim', 'quoted', 'recomputed', 'tolerance'])
def status(row):
    if pd.isna(row['tolerance']):                    # quoted as an upper bound, e.g. "p < 0.001"
        return 'OK' if row['recomputed'] < row['quoted'] else 'MISMATCH'
    return 'OK' if abs(row['recomputed'] - row['quoted']) <= row['tolerance'] + 1e-12 else 'MISMATCH'
table['status'] = table.apply(status, axis=1)
table['recomputed'] = table['recomputed'].astype(float).round(4)
(STORE / 'reports').mkdir(exist_ok=True)
table.to_csv(STORE / 'reports' / 'verification.csv', index=False)
print(f"{(table['status'] == 'OK').sum()} OK, {(table['status'] == 'MISMATCH').sum()} MISMATCH of {len(table)} checks")
display(table)

62 OK, 0 MISMATCH of 62 checks


,id,quoted in,claim,quoted,recomputed,tolerance,status
0,I01,plan 14,reimplementation: successful runs (64 x 2 modes x 3 seeds),384.000,384.0000,0.0000,OK
1,I02,plan 14,reimplementation: failed runs,0.000,0.0000,0.0000,OK
2,I03,reproduce M3,reproduction: methods with 64 successful series,8.000,8.0000,0.0000,OK
3,I04,reimplement 12,M6-C: successful runs (8 configs x 9 MGAB x 3 seeds),216.000,216.0000,0.0000,OK
4,I05,reimplement 13,M8: rescored models (2 modes x 64 x seeds 47-48),256.000,256.0000,0.0000,OK
5,I06,reimplement 14,M9: analysed models,192.000,192.0000,0.0000,OK
6,I07,reimplement 14,M9 events: rows with peak < mean percentile (0 = file not corrupted; use results_fix/ ...,0.000,0.0000,0.0000,OK
7,I08,reimplement 14,M9 events: missing OE values (0 = file not corrupted),0.000,0.0000,0.0000,OK
8,I09,reimplement 15,M10: analysed models,192.000,192.0000,0.0000,OK
9,I10,reimplement 15,M10: gradient runs (9 MGAB x 2 variants),18.000,18.0000,0.0000,OK
